In [66]:
# autoreload
%load_ext autoreload
%autoreload 2

from openai import OpenAI

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

import sys
sys.path.append("../src")
from rag_helper import RAGBase

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Retrieve the data from the GitHub repository and parse it into a list of documents

In [23]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [24]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [25]:
# How many documents were retrieved and parsed?
len(documents)

72

### 2. Create an index with minsearch

In [26]:
# Instantiate the index
index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"],)
# Fit the index with the documents
index.fit(documents)

In [27]:
def search(query, num_results=5, filter_dict=None, boost_dict=None):
    """
    Search the index for a given query.

    Args:
        query (str): The search query.
        num_results (int): The number of results to return. Default is 5.
        filter_dict (dict): A dictionary of filters to apply to the search. Default is None.
        boost_dict (dict): A dictionary of fields to boost in the search. Default is None

    Returns:
        list: A list of search results.
    """
    return index.search(
        query,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )

In [28]:
question = "How does the agentic loop keep calling the model until it stops?"
results = index.search(question, num_results=5)

In [29]:
# What is the file name of the first result?
results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

### 3. Create a rag from this index

In [41]:
# Instatiate openai client
llm_client = OpenAI()

In [63]:
# Instantiate RAG helper
assistant = RAGBase(index=index, llm_client=llm_client)

[INIT] Initializing RAGBase
[INIT] RAGBase initialized with model: gpt-5.4-mini


In [64]:
results = assistant.rag(
    query=question)

[SEARCH] Query: How does the agentic loop keep calling the model until it stops?, Num Results: 5, Boost: None, Filter: None
[SEARCH] Found 5 results
[BUILD_CONTEXT] Building context from 5 search results
[BUILD_CONTEXT] Context built with length: 29802


In [65]:
results

('The loop keeps calling the model by using a `while True` loop and checking whether the model returned any `function_call` items.\n\n- It sends the current `messages` to the model.\n- If the response contains function calls, it runs those tools, appends the tool outputs to `messages`, and loops again.\n- If the response has no function calls, it breaks out of the loop and stops.\n\nSo the stop condition is: **no function calls in the latest response**.',
 {'input_tokens': 7121,
  'cached_tokens': 6912,
  'output_tokens': 103,
  'reasoning_tokens': 0,
  'total_tokens': 7224})

### 4. Chunking and reindexing the documents

In [67]:
# Create chunks of the documents for reindexing
chunks = chunk_documents(documents, size=2000, step=1000)

In [68]:
# Index chunks with minsearch
index_chunks = Index(
    text_fields = ["content"],
    keyword_fields = ["filename", "start"],)
index_chunks.fit(chunks)

In [71]:
# How many chunks were created from the documents?
len(chunks)

295

### 5. RAG with chunks

In [72]:
# Instantiate RAG helper
assistant = RAGBase(index=index_chunks, llm_client=llm_client)

[INIT] Initializing RAGBase
[INIT] RAGBase initialized with model: gpt-5.4-mini


In [73]:
results = assistant.rag(
    query=question)

[SEARCH] Query: How does the agentic loop keep calling the model until it stops?, Num Results: 5, Boost: None, Filter: None
[SEARCH] Found 5 results
[BUILD_CONTEXT] Building context from 5 search results
[BUILD_CONTEXT] Context built with length: 9638


In [74]:
results

('It keeps running a `while True` loop and checks each model response for any `function_call` items.\n\n- If the response includes a function call, the code runs the tool, appends the result to `messages`, and sets `has_function_calls = True`.\n- If there are no function calls in that turn, it breaks out of the loop.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn short: it keeps calling the model until the model returns a response with no more tool calls.',
 {'input_tokens': 2304,
  'cached_tokens': 0,
  'output_tokens': 117,
  'reasoning_tokens': 0,
  'total_tokens': 2421})